# Lab 1 — NumPy (Numerical Python)

## What is NumPy?
**NumPy** is the core Python library for numerical and scientific computing. Its central object is the **`ndarray`** (N‑dimensional array): a grid of values that all have the **same data type**, stored in **one contiguous block of memory**.

Almost every data‑science / ML library is built on top of NumPy — **Pandas, SciPy, Matplotlib, scikit‑learn, OpenCV, TensorFlow and PyTorch** all either use NumPy arrays directly or copy its API.

## Why do we need NumPy? (Python list vs NumPy array)

| Feature | Python `list` | NumPy `ndarray` |
|---|---|---|
| Data type | Mixed types allowed (each item is a full Python object) | Single fixed type (`int32`, `float32`, …) |
| Memory | Array of pointers to objects scattered in memory → large | Contiguous raw bytes → compact |
| Speed | Loops run in the Python interpreter → slow | Operations run in compiled C code → **10–100× faster** |
| Math | `[1,2] * 2` → `[1,2,1,2]` (repetition) | `np.array([1,2]) * 2` → `[2,4]` (element‑wise math) |
| Multi‑dimensional | Lists of lists, awkward | Native N‑D support with `shape`, `axis`, slicing |
| Built‑in functions | Very few | Hundreds: statistics, linear algebra, FFT, random, … |

Key ideas that make NumPy fast and convenient:
* **Vectorization** – write `a + b` instead of a `for` loop; the loop happens in C.
* **Broadcasting** – arrays of different shapes can be combined automatically (e.g. add a row vector to every row of a matrix).
* **Views instead of copies** – slicing/reshaping usually does not copy data, saving memory.

## Applications of NumPy
* **Machine Learning / Deep Learning** – datasets, feature matrices, weights and gradients are arrays.
* **Image processing** – an image is a `(height, width, channels)` array of pixel values.
* **Signal & audio processing** – 1‑D arrays of samples, Fourier transforms (`np.fft`).
* **Statistics & data analysis** – mean, variance, percentiles, correlation; Pandas is built on it.
* **Linear algebra** – matrix multiplication, inverse, determinant, eigenvalues, solving equations.
* **Simulation / scientific computing** – physics, finance (Monte‑Carlo with `np.random`), engineering.

## Important terms
* **`ndim`** – number of dimensions (axes). A vector has 1, a matrix has 2.
* **`shape`** – tuple giving the size along each axis, e.g. `(2, 3)` = 2 rows, 3 columns.
* **`size`** – total number of elements (product of the shape).
* **`dtype`** – data type of the elements (`int64`, `float32`, `bool`, …).
* **axis** – `axis=0` runs **down the rows** (vertically), `axis=1` runs **across the columns** (horizontally).


## 1. Creating an array and inspecting its attributes

`np.array()` converts a Python list (or nested list) into a NumPy array.

**Code explanation**
* `import numpy as np` – imports NumPy with the standard alias `np`.
* `[[1,2,3],[4,5,6]]` – a nested list → a **2‑D array** with 2 rows and 3 columns.
* `dtype=np.float32` – forces each element to be a 32‑bit float (4 bytes). Without it, NumPy would infer `int64`. `float32` is common in ML/GPUs because it uses half the memory of `float64`.
* `a.ndim` → `2` (rows and columns), `a.shape` → `(2, 3)`, `a.dtype` → `float32`.


In [1]:
import numpy as np
a = np.array([[1,2,3],[4,5,6]],dtype=np.float32)
print (a.ndim, a.shape, a.dtype)

2 (2, 3) float32


## 2. Creating arrays filled with ones and zeros

Very often we need an array of a known size before filling it with values (e.g. initialising weights, masks, or result buffers).

* `np.ones(shape, dtype)` – every element is `1`.
* `np.zeros(shape, dtype)` – every element is `0`.
* The **shape is passed as a tuple** `(3, 5)` → 3 rows × 5 columns.

Related functions: `np.full(shape, value)`, `np.eye(n)` (identity matrix), `np.empty(shape)` (uninitialised – fastest, but contains garbage), `np.ones_like(x)` / `np.zeros_like(x)` (same shape as an existing array).


In [2]:
a = np.ones ((3,5),dtype=np.float32)
b = np.zeros ((3,5),dtype=np.float32)

print(a)
print(b)

[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


## 3. Creating a range of numbers with `np.arange`

`np.arange(start, stop, step)` works like Python's `range()` but returns a NumPy array.

* `np.arange(3, 15)` → integers from **3 up to 14** (the `stop` value **15 is excluded**).
* Default `step` is 1; it can also be a float, e.g. `np.arange(0, 1, 0.25)` → `[0, 0.25, 0.5, 0.75]`.
* `type(c)` shows the result is a `numpy.ndarray`, not a list.

> Tip: for floating‑point ranges prefer `np.linspace(start, stop, num)` (shown later), which gives an exact number of points and includes the end point.


In [3]:
c = np.arange (3,15)

print(c, type(c))

[ 3  4  5  6  7  8  9 10 11 12 13 14] <class 'numpy.ndarray'>


## 4. Joining arrays — `np.concatenate` (default `axis=0`)

`np.concatenate([arr1, arr2, ...], axis=0)` joins arrays **along an existing axis**.

* Here `a` is a `(3,5)` array of ones and `b` is a `(3,5)` array of zeros (from step 2).
* With the default **`axis=0`** the arrays are stacked **vertically** (rows of `b` placed below rows of `a`).
* Result shape: `(3+3, 5)` = **`(6, 5)`**.
* Rule: all dimensions **except the joining axis** must match.


In [4]:
res = np.concatenate([a,b])

print(res, res.ndim, res.shape, res.dtype)

[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]] 2 (6, 5) float32


## 5. Concatenating along `axis=1` (side by side)

* With **`axis=1`** the arrays are joined **horizontally** – columns of `b` are appended to the right of `a`.
* Result shape: `(3, 5+5)` = **`(3, 10)`**.
* Shortcuts: `np.vstack([a,b])` ≡ `axis=0`, `np.hstack([a,b])` ≡ `axis=1`.

> Note: the print statement uses `res.dtype` instead of `res1.dtype`; it still shows `float32` because both arrays have the same type.


In [5]:
res1  = np.concatenate([a,b],axis=1)

print(res1, res1.ndim, res1.shape, res.dtype)

[[1. 1. 1. 1. 1. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 0. 0. 0. 0. 0.]] 2 (3, 10) float32


## 6. Random arrays — `np.random.random`

* `np.random.random((10, 3))` creates a `10 × 3` array of random floats drawn **uniformly from [0.0, 1.0)**.
* Output changes every run. To make results reproducible use a **seed** (see the extra section on random numbers at the end).
* Uses: initialising neural‑network weights, creating test data, simulations, random sampling / shuffling of datasets.


In [6]:
nparray = np.random.random((10,3))

print(nparray)

[[0.62436667 0.07120768 0.67595053]
 [0.39551078 0.08188835 0.23325959]
 [0.54094584 0.69664544 0.98920719]
 [0.80004932 0.00602269 0.08008121]
 [0.17853736 0.76202549 0.01115447]
 [0.85407611 0.98461175 0.15263245]
 [0.57504109 0.5577419  0.05004213]
 [0.30646532 0.09303459 0.99189309]
 [0.14973019 0.27870168 0.6310096 ]
 [0.00719556 0.08982006 0.40266552]]


## 7. Changing the shape — `reshape` and `ravel`

**Reshaping** changes how the same data is arranged into rows/columns **without changing the data**. The total number of elements must stay the same (here 6).

**Code explanation**
* `np.array([1,2,3,4,5,6])` – a 1‑D array of shape `(6,)`.
* `a.reshape(3, 2)` – 3 rows × 2 columns (3×2 = 6 ✔). Elements are filled **row by row** (C order).
* `a.reshape(2, -1)` – **`-1` means "work it out for me"**: NumPy computes 6 / 2 = 3 → shape `(2, 3)`.
* `a.ravel()` – **flattens** any N‑D array back into 1‑D. (`ravel` returns a view when possible; `a.flatten()` always returns a copy.)

Uses: converting an image `(28, 28)` into a feature vector `(784,)` for ML, or batching data.


In [7]:
a = np.array([1,2,3,4,5,6])
print("before reshape :\n" , a)
a = a.reshape(3,2)
print("after reshape: \n" ,a)
a = a.reshape(2,-1)
print(a)
a = a.ravel()
print(a)

before reshape :
 [1 2 3 4 5 6]
after reshape: 
 [[1 2]
 [3 4]
 [5 6]]
[[1 2 3]
 [4 5 6]]
[1 2 3 4 5 6]


## 8. Transpose — swapping axes

The **transpose** turns rows into columns and columns into rows.

**Code explanation**
* `np.arange(10).reshape(5, 2)` – numbers 0‑9 arranged as 5 rows × 2 columns.
* `a.T` – shortcut for transpose → shape becomes `(2, 5)`.
* `a.transpose((1, 0))` – general form: the tuple tells the **new order of axes**. `(1, 0)` means "new axis 0 = old axis 1, new axis 1 = old axis 0", so it swaps them back to `(5, 2)`.
* For 3‑D data, e.g. an image stored as `(channels, height, width)`, `img.transpose((1, 2, 0))` converts it to `(height, width, channels)`.

Transpose is essential in linear algebra (e.g. $X^T X$ in linear regression).


In [8]:
a = np.arange(10).reshape(5,2) 
print(a)
a = a.T 
print(a)
a = a.transpose((1,0))
print(a)

[[0 1]
 [2 3]
 [4 5]
 [6 7]
 [8 9]]
[[0 2 4 6 8]
 [1 3 5 7 9]]
[[0 1]
 [2 3]
 [4 5]
 [6 7]
 [8 9]]


## 9. Slicing and summing along axes

### Slicing syntax: `a[start:stop:step]`
* `start` is **included**, `stop` is **excluded**, `step` defaults to 1.
* Negative indices count from the end (`-1` is the last element).

| Expression | Meaning | Result for `range(10)` |
|---|---|---|
| `a[:3]` | first 3 elements | indices 0, 1, 2 |
| `a[-3:]` | last 3 elements | indices 7, 8, 9 |
| `a[3:8:2]` | from 3 to 7, every 2nd | indices 3, 5, 7 |
| `a[4:1:-1]` | backwards from 4 down to 2 | indices 4, 3, 2 |

> Note on the code below: the first line `a[:3]` is not stored back into `a`, so `print(a)` shows the full list. Write `a = a[:3]` to see `[0, 1, 2]`. These slices use a Python **list**; the same syntax works on NumPy arrays (and also extends to multiple dimensions, e.g. `m[1:3, ::2]`).

### Summation with `axis`
* `a.sum()` – sum of **all** elements.
* `a.sum(axis=0)` – sum **down the rows** (one result per column). For a 1‑D array there is only axis 0, so it is the same as the total: `0+1+…+9 = 45`.
* `keepdims=True` – keeps the reduced axis with size 1, giving `array([45])` instead of `45`. This keeps shapes compatible for broadcasting (e.g. `x / x.sum(axis=1, keepdims=True)` to normalise rows).
* Only the **last expression** of a Jupyter cell is displayed automatically, which is why only `array([45])` appears as output.


In [9]:
a = list(range(10)) 
a[:3] 
print(a)
# indices 0, 1, 2 
a = list(range(10))
a=a[-3:] 
print(a)
# indices 7, 8, 9 
a = list(range(10))
a=a[3:8:2] 
print(a)
# indices 3, 5, 7 
a = list(range(10))
a=a[4:1:-1]
print(a)

# sum all entries a.sum(axis=0) 
a = np.arange(10)
# Axes a.sum()
# sum over rows 
a.sum(axis=0) 
# sum over columns 
a.sum(axis=0, keepdims=True)



[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[7, 8, 9]
[3, 5, 7]
[4, 3, 2]


array([45])

### 9.1 Summing a 2‑D array over rows / columns (as in the lab manual)

The manual uses `a.sum(axis=1)` and `a.sum(axis=1, keepdims=True)`. These need a **2‑D** array (a 1‑D array has no axis 1 and would raise `AxisError`), so here the same numbers 0‑9 are reshaped into `(2, 5)`.

* `a.sum()` → sum of everything.
* `a.sum(axis=0)` → collapses the rows → one total **per column** → shape `(5,)`.
* `a.sum(axis=1)` → collapses the columns → one total **per row** → shape `(2,)`.
* `a.sum(axis=1, keepdims=True)` → same totals but shape `(2, 1)`, so it can be broadcast back against `a` (e.g. to compute each element's share of its row total).


In [10]:
a = np.arange(10).reshape(2, 5)
print(a)
print("sum all            :", a.sum())
print("axis=0 (per column):", a.sum(axis=0))
print("axis=1 (per row)   :", a.sum(axis=1), a.sum(axis=1).shape)
row_tot = a.sum(axis=1, keepdims=True)
print("keepdims=True      :\n", row_tot, row_tot.shape)
print("share of row total :\n", (a / row_tot).round(2))

[[0 1 2 3 4]
 [5 6 7 8 9]]
sum all            : 45
axis=0 (per column): [ 5  7  9 11 13]
axis=1 (per row)   : [10 35] (2,)
keepdims=True      :
 [[10]
 [35]] (2, 1)
share of row total :
 [[0.   0.1  0.2  0.3  0.4 ]
 [0.14 0.17 0.2  0.23 0.26]]


## 10. Slicing images (multi‑dimensional indexing)

A colour image is a 3‑D array of shape **`(height, width, channels)`**. Slicing lets us crop, flip and edit it without loops.

* `I[1:-1, 1:-1]` – drop the first and last row and column → removes a **one‑pixel border**. (Channels are untouched because they are not mentioned.)
* `I[:, :, ::-1]` – reverse the **channel axis**: RGB ⇄ BGR (OpenCV loads images as BGR, Matplotlib expects RGB).
* `I[I < 10] = 0` – **boolean masking**: set every pixel value below 10 to 0 (e.g. remove dark noise).
* `I[[1, 3], :]` – **fancy indexing**: select rows 1 and 3 only.

> The original cell was incomplete (`I=[1:-1,1:-1]` is a syntax error because `I` was never defined as an array). Below it is rewritten to run on a small sample "image" made with `np.random.randint`.


In [11]:
# a sample 6x6 RGB "image" with pixel values 0-255
I = np.random.randint(0, 256, size=(6, 6, 3), dtype=np.uint8)
print("original shape:", I.shape)

I = I[1:-1, 1:-1]        # select all but one-pixel border
print("after cropping border:", I.shape)

I = I[:, :, ::-1]        # swap channel order (RGB <-> BGR)

I[I < 10] = 0            # set dark pixels to black (boolean mask)

rows = I[[1, 3], :]      # select 2nd and 4th row (fancy indexing)
print("selected rows shape:", rows.shape)

original shape: (6, 6, 3)
after cropping border: (4, 4, 3)
selected rows shape: (2, 4, 3)


---
# Additional NumPy Topics (not covered above)

The sections below complete the important NumPy concepts needed for data‑science labs and exams.


## 11. More ways to create arrays

* `np.linspace(start, stop, num)` – `num` **evenly spaced** values, **including** `stop`. Useful for plotting smooth curves.
* `np.full(shape, value)` – array filled with a constant.
* `np.eye(n)` – `n × n` **identity matrix**.
* `np.zeros_like(x)` – zeros with the same shape and dtype as `x`.
* Extra attributes: `size` (number of elements), `itemsize` (bytes per element), `nbytes` (total bytes).


In [12]:
x = np.linspace(0, 1, 5)
print("linspace:", x)

print("full:\n", np.full((2, 3), 7))
print("identity:\n", np.eye(3))
print("zeros_like:", np.zeros_like(x))

m = np.arange(12).reshape(3, 4)
print("size =", m.size, "| itemsize =", m.itemsize, "bytes | nbytes =", m.nbytes, "bytes")

linspace: [0.   0.25 0.5  0.75 1.  ]
full:
 [[7 7 7]
 [7 7 7]]
identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
zeros_like: [0. 0. 0. 0. 0.]
size = 12 | itemsize = 8 bytes | nbytes = 96 bytes


## 12. Indexing 2‑D arrays

* `m[row, col]` – single element.
* `m[row]` or `m[row, :]` – a whole row; `m[:, col]` – a whole column.
* `m[r1:r2, c1:c2]` – a sub‑matrix (block).
* Indexing starts at **0**.


In [13]:
m = np.arange(1, 13).reshape(3, 4)
print(m)
print("element [1,2]  :", m[1, 2])
print("row 0          :", m[0])
print("column 1       :", m[:, 1])
print("sub-matrix     :\n", m[0:2, 1:3])
print("last row, reversed:", m[-1, ::-1])

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
element [1,2]  : 7
row 0          : [1 2 3 4]
column 1       : [ 2  6 10]
sub-matrix     :
 [[2 3]
 [6 7]]
last row, reversed: [12 11 10  9]


## 13. Boolean indexing and `np.where`

A comparison like `a > 5` returns a **boolean array** of the same shape. Using it as an index **filters** the array.

* `a[a > 5]` – elements greater than 5.
* Combine conditions with `&` (and), `|` (or), `~` (not) and **parentheses**: `a[(a > 2) & (a < 8)]`.
* `np.where(cond, x, y)` – element‑wise *if‑else*: picks from `x` where `cond` is True, otherwise from `y`.

Very common in data cleaning, e.g. replacing negative values or outliers.


In [14]:
a = np.arange(10)
print("mask a > 5  :", a > 5)
print("a[a > 5]    :", a[a > 5])
print("2 < a < 8   :", a[(a > 2) & (a < 8)])
print("even numbers:", a[a % 2 == 0])

print("np.where    :", np.where(a < 5, 0, a))     # replace values < 5 with 0
print("indices     :", np.where(a > 6))           # positions where condition is True

mask a > 5  : [False False False False False False  True  True  True  True]
a[a > 5]    : [6 7 8 9]
2 < a < 8   : [3 4 5 6 7]
even numbers: [0 2 4 6 8]
np.where    : [0 0 0 0 0 5 6 7 8 9]
indices     : (array([7, 8, 9]),)


## 14. Element‑wise (vectorized) arithmetic

Arithmetic operators work **element by element** on arrays of the same shape, with no loops needed.

* `+ - * / ** %` all work element‑wise.
* **`*` is NOT matrix multiplication**; use `@` or `np.dot` for that (see Linear Algebra).
* **Universal functions (ufuncs)** like `np.sqrt`, `np.exp`, `np.log`, `np.sin`, `np.abs` apply to every element.


In [15]:
x = np.array([1, 2, 3, 4])
y = np.array([10, 20, 30, 40])

print("x + y  :", x + y)
print("x * y  :", x * y)
print("y / x  :", y / x)
print("x ** 2 :", x ** 2)
print("sqrt(x):", np.sqrt(x))
print("exp(x) :", np.exp(x))
print("log(y) :", np.log(y))

x + y  : [11 22 33 44]
x * y  : [ 10  40  90 160]
y / x  : [10. 10. 10. 10.]
x ** 2 : [ 1  4  9 16]
sqrt(x): [1.         1.41421356 1.73205081 2.        ]
exp(x) : [ 2.71828183  7.3890561  20.08553692 54.59815003]
log(y) : [2.30258509 2.99573227 3.40119738 3.68887945]


## 15. Vectorization vs Python loop — speed comparison

This demonstrates *why* NumPy is needed: summing the squares of 1 million numbers with a Python loop versus a single NumPy expression.


In [16]:
import time

n = 1_000_000
lst = list(range(n))
arr = np.arange(n)

t = time.time()
s1 = sum(v * v for v in lst)
loop_time = time.time() - t

t = time.time()
s2 = np.sum(arr * arr)
numpy_time = time.time() - t

print(f"Python loop : {loop_time:.4f} s")
print(f"NumPy       : {numpy_time:.4f} s")
print(f"NumPy is about {loop_time / numpy_time:.0f}x faster, same result: {s1 == s2}")

Python loop : 0.0454 s
NumPy       : 0.0125 s
NumPy is about 4x faster, same result: True


## 16. Broadcasting

**Broadcasting** lets NumPy operate on arrays of **different shapes** by virtually "stretching" the smaller one, without copying data.

**Rules** (compare shapes from the **right**): two dimensions are compatible if they are **equal** or one of them is **1**.

| A shape | B shape | Result |
|---|---|---|
| `(3, 4)` | scalar | `(3, 4)` |
| `(3, 4)` | `(4,)` | `(3, 4)` (row added to every row) |
| `(3, 4)` | `(3, 1)` | `(3, 4)` (column added to every column) |
| `(3, 1)` | `(1, 4)` | `(3, 4)` |
| `(3, 4)` | `(3,)` | ❌ error (4 vs 3) |

Typical use: **feature normalisation** `(X - X.mean(axis=0)) / X.std(axis=0)`.


In [17]:
m = np.ones((3, 4))
row = np.array([1, 2, 3, 4])          # shape (4,)
col = np.array([[10], [20], [30]])    # shape (3, 1)

print("m + 5:\n", m + 5)
print("m + row:\n", m + row)
print("m + col:\n", m + col)
print("col + row (3,1)+(4,) -> (3,4):\n", col + row)

m + 5:
 [[6. 6. 6. 6.]
 [6. 6. 6. 6.]
 [6. 6. 6. 6.]]
m + row:
 [[2. 3. 4. 5.]
 [2. 3. 4. 5.]
 [2. 3. 4. 5.]]
m + col:
 [[11. 11. 11. 11.]
 [21. 21. 21. 21.]
 [31. 31. 31. 31.]]
col + row (3,1)+(4,) -> (3,4):
 [[11 12 13 14]
 [21 22 23 24]
 [31 32 33 34]]


## 17. Stacking and splitting arrays

* `np.vstack` – stack vertically (same as `concatenate(axis=0)`).
* `np.hstack` – stack horizontally (same as `concatenate(axis=1)`).
* `np.stack` – join along a **new** axis (e.g. two `(3,)` arrays → `(2, 3)`).
* `np.split(arr, n, axis)` – the reverse: divide an array into `n` equal parts. Also `np.hsplit`, `np.vsplit`.


In [18]:
p = np.array([1, 2, 3])
q = np.array([4, 5, 6])

print("vstack:\n", np.vstack([p, q]))
print("hstack:", np.hstack([p, q]))
print("stack axis=1:\n", np.stack([p, q], axis=1))

big = np.arange(12).reshape(3, 4)
left, right = np.hsplit(big, 2)
print("left half:\n", left)
print("right half:\n", right)
print("split into 3 parts:", np.split(np.arange(9), 3))

vstack:
 [[1 2 3]
 [4 5 6]]
hstack: [1 2 3 4 5 6]
stack axis=1:
 [[1 4]
 [2 5]
 [3 6]]
left half:
 [[0 1]
 [4 5]
 [8 9]]
right half:
 [[ 2  3]
 [ 6  7]
 [10 11]]
split into 3 parts: [array([0, 1, 2]), array([3, 4, 5]), array([6, 7, 8])]


## 18. Adding / removing dimensions

* `np.newaxis` (or `None`) inserts an axis of size 1: `(3,)` → `(1, 3)` or `(3, 1)`.
* `np.expand_dims(a, axis)` – same, as a function.
* `np.squeeze(a)` – removes all axes of size 1.

Needed when a model expects a batch dimension, e.g. a single image `(28, 28)` → `(1, 28, 28)`.


In [19]:
v = np.array([1, 2, 3])
print("original       :", v.shape)
print("row vector     :", v[np.newaxis, :].shape)
print("column vector  :", v[:, np.newaxis].shape)
print("expand_dims    :", np.expand_dims(v, axis=0).shape)
print("squeeze (1,3,1):", np.squeeze(np.zeros((1, 3, 1))).shape)

original       : (3,)
row vector     : (1, 3)
column vector  : (3, 1)
expand_dims    : (1, 3)
squeeze (1,3,1): (3,)


## 19. Copy vs View

* **Slicing** and `reshape` return a **view** – it shares memory with the original, so changing the view **changes the original**.
* `.copy()` makes an independent **deep copy**.
* Fancy / boolean indexing always returns a **copy**.

This is a very common source of bugs, so remember to use `.copy()` when you want to modify data safely.


In [20]:
a = np.arange(5)
view = a[1:4]
view[0] = 100
print("after modifying view, original a:", a)     # a changed!

a = np.arange(5)
cp = a[1:4].copy()
cp[0] = 100
print("after modifying copy, original a:", a)     # a unchanged
print("shares memory?", np.shares_memory(a, view), np.shares_memory(a, cp))

after modifying view, original a: [  0 100   2   3   4]
after modifying copy, original a: [0 1 2 3 4]
shares memory? False False


## 20. Data types and type conversion

* Every array has one `dtype`. Common ones: `int32`, `int64`, `float32`, `float64`, `bool`, `uint8` (images, 0‑255), `complex128`.
* `astype(new_dtype)` converts and returns a **new** array. Converting float → int **truncates** decimals.
* Choosing smaller dtypes saves memory (important for large datasets).


In [21]:
f = np.array([1.7, 2.2, -3.9])
print(f.dtype, "->", f.astype(np.int32), f.astype(np.int32).dtype)
print("bool conversion:", np.array([0, 1, 2]).astype(bool))

big64 = np.ones(1000, dtype=np.float64)
print("float64 bytes:", big64.nbytes, "| float32 bytes:", big64.astype(np.float32).nbytes)

float64 -> [ 1  2 -3] int32
bool conversion: [False  True  True]
float64 bytes: 8000 | float32 bytes: 4000


## 21. Linear algebra (`np.linalg`)

| Operation | Code |
|---|---|
| Matrix multiplication | `A @ B` or `np.dot(A, B)` |
| Transpose | `A.T` |
| Determinant | `np.linalg.det(A)` |
| Inverse | `np.linalg.inv(A)` |
| Solve $Ax = b$ | `np.linalg.solve(A, b)` |
| Eigenvalues / vectors | `np.linalg.eig(A)` |
| Rank | `np.linalg.matrix_rank(A)` |
| Norm (length) | `np.linalg.norm(v)` |

For matrix multiplication `(m × n) @ (n × p)` → `(m × p)`: the inner dimensions must match.

Example: solve the system $2x + y = 5$, $x + 3y = 10$.


In [22]:
A = np.array([[2, 1],
              [1, 3]])
b = np.array([5, 10])

print("A @ A:\n", A @ A)
print("A * A (element-wise, NOT matrix product):\n", A * A)
print("determinant :", np.linalg.det(A).round(2))
print("inverse     :\n", np.linalg.inv(A))
print("solution x, y:", np.linalg.solve(A, b))

vals, vecs = np.linalg.eig(A)
print("eigenvalues :", vals.round(3))
print("rank        :", np.linalg.matrix_rank(A))
print("norm of b   :", np.linalg.norm(b).round(3))

A @ A:
 [[ 5  5]
 [ 5 10]]
A * A (element-wise, NOT matrix product):
 [[4 1]
 [1 9]]
determinant : 5.0
inverse     :
 [[ 0.6 -0.2]
 [-0.2  0.4]]
solution x, y: [1. 3.]
eigenvalues : [1.382 3.618]


rank        : 2
norm of b   : 11.18


## 22. Random numbers in detail

* **Seed** makes random results **reproducible** (same numbers every run) – important for experiments.
* The modern, recommended API is `rng = np.random.default_rng(seed)`.

| Function | Produces |
|---|---|
| `rng.random(size)` | uniform floats in [0, 1) |
| `rng.integers(low, high, size)` | random integers in [low, high) |
| `rng.normal(mean, std, size)` | Gaussian (normal) distribution |
| `rng.uniform(low, high, size)` | uniform floats in [low, high) |
| `rng.choice(arr, size, replace)` | random sample from an array |
| `rng.permutation(arr)` / `rng.shuffle(arr)` | shuffled order |

Older equivalent style: `np.random.seed(42)`, `np.random.randint`, `np.random.randn`, `np.random.rand`.


In [23]:
rng = np.random.default_rng(42)

print("uniform   :", rng.random(3).round(3))
print("integers  :", rng.integers(1, 7, size=5))            # dice rolls
data = rng.normal(loc=50, scale=10, size=1000)             # mean 50, std 10
print("normal    : mean=%.2f std=%.2f" % (data.mean(), data.std()))
print("choice    :", rng.choice(["A", "B", "C"], size=4))
print("permutation:", rng.permutation(10))

# train/test split of indices (typical ML use)
idx = rng.permutation(10)
print("train idx:", idx[:8], "| test idx:", idx[8:])

uniform   : [0.774 0.439 0.859]
integers  : [1 5 2 1 4]
normal    : mean=49.73 std=9.85
choice    : ['C' 'A' 'C' 'A']
permutation: [2 3 7 5 1 4 9 8 6 0]
train idx: [1 7 9 4 5 6 2 0] | test idx: [8 3]


## 23. Sorting, searching and unique values

* `np.sort(a)` – returns a sorted copy (`a.sort()` sorts in place).
* `np.argsort(a)` – indices that would sort the array (useful for ranking).
* `np.unique(a, return_counts=True)` – distinct values and how often they occur.
* `np.isin(a, values)` – membership test.
* `np.clip(a, lo, hi)` – limit values into a range.


In [24]:
a = np.array([3, 1, 4, 1, 5, 9, 2, 6, 5, 3])

print("sorted      :", np.sort(a))
print("descending  :", np.sort(a)[::-1])
print("argsort     :", np.argsort(a))
vals, counts = np.unique(a, return_counts=True)
print("unique      :", vals)
print("counts      :", counts)
print("isin [1, 5] :", np.isin(a, [1, 5]))
print("clip 2..5   :", np.clip(a, 2, 5))

m = np.array([[3, 1], [2, 5], [1, 4]])
print("sort each row:\n", np.sort(m, axis=1))

sorted      : [1 1 2 3 3 4 5 5 6 9]
descending  : [9 6 5 5 4 3 3 2 1 1]
argsort     : [1 3 6 0 9 2 8 4 7 5]
unique      : [1 2 3 4 5 6 9]
counts      : [2 1 2 1 2 1 1]
isin [1, 5] : [False  True False  True  True False False False  True False]
clip 2..5   : [3 2 4 2 5 5 2 5 5 3]
sort each row:
 [[1 3]
 [2 5]
 [1 4]]


## 24. Saving and loading arrays

* `np.save("file.npy", arr)` / `np.load("file.npy")` – fast binary format for a single array.
* `np.savez("file.npz", x=a, y=b)` – several arrays in one file.
* `np.savetxt("file.csv", arr, delimiter=",")` / `np.loadtxt(...)` / `np.genfromtxt(...)` – human‑readable text / CSV (`genfromtxt` also handles missing values).


In [25]:
import os, tempfile

folder = tempfile.mkdtemp()
arr = np.arange(6).reshape(2, 3)

np.save(os.path.join(folder, "arr.npy"), arr)
print("loaded .npy:\n", np.load(os.path.join(folder, "arr.npy")))

np.savetxt(os.path.join(folder, "arr.csv"), arr, delimiter=",", fmt="%d")
print("loaded .csv:\n", np.loadtxt(os.path.join(folder, "arr.csv"), delimiter=","))

loaded .npy:
 [[0 1 2]
 [3 4 5]]
loaded .csv:
 [[0. 1. 2.]
 [3. 4. 5.]]


---
# Part 3 — Extra NumPy Essentials

Topics that are not part of this lab and are not covered in the later experiments either, but come up constantly in day-to-day and professional NumPy work.


## 25. Everyday array utilities

| Function | Purpose |
|---|---|
| `np.any`, `np.all` | is any / every element True? (data validation) |
| `np.append`, `np.insert`, `np.delete` | add / insert / remove elements (return a **new** array) |
| `np.tile`, `np.repeat` | repeat a whole array / repeat each element |
| `np.flip`, `np.roll`, `np.rot90` | reverse, circular shift, rotate (image augmentation) |
| `np.argmax`, `np.unravel_index` | position of the maximum in a 2‑D array |
| `np.round`, `np.floor`, `np.ceil` | rounding |
| `np.union1d`, `np.intersect1d`, `np.setdiff1d` | set operations (e.g. common customer IDs) |
| `np.count_nonzero` | count True / non‑zero values |
| `np.column_stack` | build a feature matrix from 1‑D columns |


In [26]:
a = np.array([3, 8, 1, 9, 4])
print("any > 8:", np.any(a > 8), "| all > 0:", np.all(a > 0), "| count > 3:", np.count_nonzero(a > 3))
print("append :", np.append(a, [10, 11]))
print("insert :", np.insert(a, 1, 100))
print("delete :", np.delete(a, [0, 2]))
print("tile   :", np.tile([1, 2], 3))
print("repeat :", np.repeat([1, 2], 3))
print("flip   :", np.flip(a))
print("roll 2 :", np.roll(a, 2))

m = np.array([[1, 2], [3, 4]])
print("rot90:\n", np.rot90(m))
big = np.array([[5, 1, 7], [9, 2, 3]])
print("position of max in 2-D:", np.unravel_index(np.argmax(big), big.shape))

x = np.array([2.45, -1.55, 3.5])
print("round/floor/ceil:", np.round(x, 1), np.floor(x), np.ceil(x))

jan = np.array([101, 102, 103, 104])
feb = np.array([103, 104, 105])
print("union:", np.union1d(jan, feb), "| common:", np.intersect1d(jan, feb), "| left in Jan:", np.setdiff1d(jan, feb))
print("column_stack:\n", np.column_stack([[1, 2, 3], [4, 5, 6]]))

any > 8: True | all > 0: True | count > 3: 3
append : [ 3  8  1  9  4 10 11]
insert : [  3 100   8   1   9   4]
delete : [8 9 4]
tile   : [1 2 1 2 1 2]
repeat : [1 1 1 2 2 2]
flip   : [4 9 1 8 3]
roll 2 : [9 4 3 8 1]
rot90:
 [[2 4]
 [1 3]]
position of max in 2-D: (np.int64(1), np.int64(0))
round/floor/ceil: [ 2.4 -1.6  3.5] [ 2. -2.  3.] [ 3. -1.  4.]


union: [101 102 103 104 105] | common: [103 104] | left in Jan: [101 102]
column_stack:
 [[1 4]
 [2 5]
 [3 6]]


## 26. Floating‑point pitfalls

Computers store decimals in binary, so tiny rounding errors appear.

* `0.1 + 0.2 == 0.3` is **False**. Compare floats with `np.isclose` / `np.allclose` (with a tolerance), never with `==`.
* Integer arrays have a **fixed size**: `int8` can hold only −128…127 and silently **overflows** (wraps around), so pick a dtype large enough.
* Division by zero gives `inf` or `nan` with a warning; `np.isfinite` detects both.


In [27]:
print("0.1 + 0.2 == 0.3    :", 0.1 + 0.2 == 0.3)
print("np.isclose(...)     :", np.isclose(0.1 + 0.2, 0.3))
print("allclose arrays     :", np.allclose([1.0, 2.0], [1.0 + 1e-10, 2.0]))

small = np.array([120, 10], dtype=np.int8)
print("int8 overflow 120+10:", small.sum(dtype=np.int8), "(wrapped around!)")

with np.errstate(divide="ignore", invalid="ignore"):
    r = np.array([1.0, 0.0, -1.0]) / 0
print("x / 0               :", r, "| finite?", np.isfinite(r))

0.1 + 0.2 == 0.3    : False
np.isclose(...)     : True
allclose arrays     : True
int8 overflow 120+10: -126 (wrapped around!)
x / 0               : [ inf  nan -inf] | finite? [False False False]


## 27. `apply_along_axis`, `vectorize` and `einsum`

* `np.apply_along_axis(func, axis, arr)` – apply your own function to every row or column.
* `np.vectorize(func)` – lets a plain Python function accept arrays. **Convenient but not faster** (still a Python loop inside); prefer real NumPy operations when possible.
* `np.einsum` – Einstein‑summation notation, one compact way to write dot products, matrix products, traces and batch operations (used a lot in deep‑learning code).


In [28]:
m = np.array([[4, 9, 2], [3, 5, 7], [8, 1, 6]])
print("range of each row:", np.apply_along_axis(lambda r: r.max() - r.min(), 1, m))

def grade(marks):
    return "Pass" if marks >= 40 else "Fail"
print("vectorize:", np.vectorize(grade)(np.array([35, 72, 40, 18])))

A = np.arange(6).reshape(2, 3)
B = np.arange(6).reshape(3, 2)
print("einsum matmul == A @ B:", np.array_equal(np.einsum("ij,jk->ik", A, B), A @ B))
print("einsum trace  :", np.einsum("ii", m), "==", np.trace(m))
print("einsum dot    :", np.einsum("i,i", [1, 2, 3], [4, 5, 6]))

range of each row: [7 4 7]
vectorize: ['Fail' 'Pass' 'Pass' 'Fail']
einsum matmul == A @ B: True
einsum trace  : 15 == 15
einsum dot    : 32


## 28. Memory and performance tips for large data

1. **Vectorize** – avoid Python `for` loops over elements.
2. Use the **smallest suitable dtype** (`float32` instead of `float64`, `uint8` for images) – halves or quarters memory.
3. Use **in‑place operations** (`a += 1`, `np.multiply(a, 2, out=a)`) to avoid creating temporary arrays.
4. **Pre‑allocate** with `np.empty` / `np.zeros` instead of growing an array with `np.append` in a loop (each append copies the whole array).
5. Prefer **views** (slicing, `reshape`) over copies when you do not need independent data.
6. For files bigger than RAM use **`np.memmap`**, which reads only the parts you access from disk.


In [29]:
import time

n = 20_000
t = time.time()
grow = np.array([])
for i in range(n):
    grow = np.append(grow, i)          # copies the array every time
append_time = time.time() - t

t = time.time()
pre = np.empty(n)
for i in range(n):
    pre[i] = i                         # writes into existing memory
prealloc_time = time.time() - t

print(f"np.append in loop : {append_time:.4f} s")
print(f"pre-allocated     : {prealloc_time:.4f} s")
print(f"np.arange (vector): instant, same result = {np.array_equal(pre, np.arange(n))}")

a = np.ones(5, dtype=np.float32)
before = id(a)
np.multiply(a, 3, out=a)               # in-place, no new array
a += 1
print("in-place result:", a, "| same object:", id(a) == before)

np.append in loop : 0.0853 s
pre-allocated     : 0.0024 s
np.arange (vector): instant, same result = True
in-place result: [4. 4. 4. 4. 4.] | same object: True


---
## Summary

| Category | Key functions |
|---|---|
| Creation | `array`, `zeros`, `ones`, `full`, `eye`, `arange`, `linspace`, `random` |
| Attributes | `ndim`, `shape`, `size`, `dtype`, `itemsize`, `nbytes` |
| Reshaping | `reshape`, `ravel`, `flatten`, `T`, `transpose`, `newaxis`, `squeeze` |
| Joining / splitting | `concatenate`, `vstack`, `hstack`, `stack`, `split` |
| Indexing | slicing `[start:stop:step]`, boolean masks, fancy indexing, `where` |
| Math | element‑wise ops, ufuncs (`sqrt`, `exp`, `log`), broadcasting |
| Linear algebra | `@`, `dot`, `linalg.inv`, `det`, `solve`, `eig`, `norm` |
| Random | `default_rng`, `random`, `integers`, `normal`, `choice`, `permutation` |
| Misc | `sort`, `argsort`, `unique`, `clip`, `save`/`load` |
| Utilities | `any`/`all`, `tile`/`repeat`, `flip`/`roll`, set ops, `isclose`, `einsum`, `out=` |

**Conclusion:** NumPy provides fast, memory‑efficient N‑dimensional arrays and vectorized operations, which makes it the foundation of numerical computing and data science in Python.
